# Deepfake Audio Detection — End-to-End Walkthrough

Classify speech as **Genuine (Human)** or **Deepfake (AI-Generated)**.

This notebook runs the full pipeline using the project's `src/` modules:
**data → log-mel features → CNN → probability**, then evaluation and a live
inference demo.

**Label convention:** `0 = Genuine`, `1 = Deepfake`. The detection score is the
deepfake probability; EER uses *deepfake* as the positive class.

**Verification targets (problem statement §4/§5):** Accuracy ≥ 80 %, EER ≤ 12 %,
F1 ≥ 80 %, per-class accuracy ≥ 75 %, and a reported confusion matrix.

## 1. Setup

This notebook runs on **Kaggle** or **locally** without edits.

- **Kaggle:** add the project code as a dataset (the folder containing `src/`) and
  add the *Fake-or-Real* audio dataset. The cell below auto-detects both paths and
  writes all outputs to `/kaggle/working` (the input mounts are read-only).
- **Local:** run from the repo root after `pip install -r requirements.txt`.

If auto-detection ever fails, set `PROJECT_ROOT` (next cell) and the
`cfg.paths.*` values (config cell) by hand.

In [ ]:
# --- Locate the project code (the folder that contains `src/`) and put it on sys.path ---
# This is the fix for `ModuleNotFoundError: No module named 'src.config'`: the parent
# of `src/` must be on sys.path *before* any `from src... import`. Keeping the path
# setup and the import in run-all order guarantees that.
import os, sys
from collections import deque

# Kaggle "Add Data" path by default; auto-detected if that doesn't exist (e.g. local).
PROJECT_ROOT = "/kaggle/input/datasets/sanjananallapati/mars-ml/deepfake-audio-detection"

_SKIP = {"real", "fake", "genuine", "spoof", "training", "validation", "testing"}

def _find_project_root(bases, max_depth=5):
    """Breadth-first search for a directory containing `src/config.py`."""
    dq = deque((b, 0) for b in bases if b and os.path.isdir(b))
    seen = set()
    while dq:
        d, depth = dq.popleft()
        if d in seen or depth > max_depth:
            continue
        seen.add(d)
        if os.path.isfile(os.path.join(d, "src", "config.py")):
            return d
        try:
            for e in os.scandir(d):
                if e.is_dir() and not e.name.startswith(".") and e.name.lower() not in _SKIP:
                    dq.append((e.path, depth + 1))
        except OSError:
            continue
    return None

if not os.path.isfile(os.path.join(PROJECT_ROOT, "src", "config.py")):
    PROJECT_ROOT = _find_project_root([os.getcwd(), "/kaggle/input", ".."]) or PROJECT_ROOT

assert os.path.isdir(os.path.join(PROJECT_ROOT, "src")), (
    f"Could not find src/ under {PROJECT_ROOT!r}. Set PROJECT_ROOT manually above."
)
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("src/ found  :", os.path.isdir(os.path.join(PROJECT_ROOT, "src")))

In [ ]:
import numpy as np
import torch

from src.config import Config
from src.utils import seed_everything, get_device, count_parameters

# Load config from the repo (NOT the kernel's cwd, which on Kaggle is /kaggle/working).
cfg = Config.load(os.path.join(PROJECT_ROOT, "config.yaml"))
seed_everything(cfg.train.seed)
device = get_device("auto")

# ── Auto-locate the Fake-or-Real splits (training / validation / testing) ────────────
_CLASS_DIRS = {"real", "fake", "genuine", "spoof", "bonafide",
               "synthetic", "human", "live", "clone"}

def _is_split_dir(path):
    """True if `path` holds class sub-folders (real/, fake/, …)."""
    try:
        return any(e.is_dir() and e.name.lower() in _CLASS_DIRS for e in os.scandir(path))
    except OSError:
        return False

def _find_data_root(bases, max_depth=6):
    """Find the directory whose `training/` sub-folder looks like a labelled split."""
    dq = deque((b, 0) for b in bases if b and os.path.isdir(b))
    seen = set()
    while dq:
        d, depth = dq.popleft()
        if d in seen or depth > max_depth:
            continue
        seen.add(d)
        names = {}
        try:
            for e in os.scandir(d):
                if e.is_dir():
                    names[e.name.lower()] = e.path
        except OSError:
            continue
        if "training" in names and _is_split_dir(names["training"]):
            return d, names
        for nm, path in names.items():
            if nm not in _CLASS_DIRS:          # don't descend into audio/class folders
                dq.append((path, depth + 1))
    return None, {}

DATA_ROOT, _splits = _find_data_root(
    [os.path.join(PROJECT_ROOT, "data"), "/kaggle/input", os.getcwd(), "."]
)
assert DATA_ROOT, (
    "Could not find the Fake-or-Real splits. Add the audio dataset, or set "
    "cfg.paths.train_dir / val_dir / test_dir manually below."
)
cfg.paths.train_dir = _splits.get("training")
cfg.paths.val_dir   = _splits.get("validation")              # None -> train.py auto-splits
cfg.paths.test_dir  = _splits.get("testing") or _splits.get("test")

# ── Send every output to a WRITABLE location (Kaggle's /kaggle/input is read-only) ───
OUT_DIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else PROJECT_ROOT
cfg.paths.model_dir   = os.path.join(OUT_DIR, "models")
cfg.paths.reports_dir = os.path.join(OUT_DIR, "reports")
cfg.paths.figures_dir = os.path.join(OUT_DIR, "reports", "figures")

print("Device     :", device, "| Torch:", torch.__version__)
print("DATA_ROOT  :", DATA_ROOT)
print("  train_dir:", cfg.paths.train_dir)
print("  val_dir  :", cfg.paths.val_dir)
print("  test_dir :", cfg.paths.test_dir)
print("Outputs -> :", OUT_DIR)

## 2. Dataset

Uses **The Fake-or-Real dataset**
(`kaggle.com/datasets/mohammedabdeldayem/the-fake-or-real-dataset`), whose splits
hold class sub-folders (`real/`, `fake/`, … — many aliases accepted):

```
…/for-norm/{training,validation,testing}/{real,fake}/*.wav
```

The previous cell already discovered and set `cfg.paths.train_dir / val_dir /
test_dir`. The cell below just scans them and reports the class balance.

> **Note on results:** the **validation** split shares speakers and synthesis
> systems with training, so very high val accuracy (often 99 %+ on FoR) is
> expected and *overstates* real-world performance. The **test** split (section 6)
> is the honest metric — and cross-dataset evaluation (e.g. ASVspoof) is stricter
> still.

In [ ]:
from src.dataset import scan_dataset, class_distribution

train_samples = scan_dataset(cfg.paths.train_dir)
print("Train distribution:", dict(class_distribution(train_samples)),
      "| total", len(train_samples))

# Validation + test are optional to scan here but useful as a sanity check:
import os
if os.path.isdir(cfg.paths.test_dir):
    test_samples = scan_dataset(cfg.paths.test_dir)
    print("Test  distribution:", dict(class_distribution(test_samples)),
          "| total", len(test_samples))

### Visualise one genuine and one deepfake clip

We plot the waveform and the **log-mel spectrogram** the model actually sees.
Deepfakes often show subtly over-smoothed spectra and unnatural harmonic/phase
structure in this time–frequency view.

In [ ]:
import matplotlib.pyplot as plt
from src.features import load_waveform, audio_path_to_feature

def first_of(label):
    for p, l in train_samples:
        if l == label:
            return p
    return None

paths = {0: first_of(0), 1: first_of(1)}
fig, axes = plt.subplots(2, 2, figsize=(12, 6))
for col, (lbl, name) in enumerate([(0, "Genuine"), (1, "Deepfake")]):
    p = paths[lbl]
    if p is None:
        continue
    y = load_waveform(p, cfg.audio)
    feat = audio_path_to_feature(p, cfg.audio, cfg.features)
    axes[0, col].plot(y); axes[0, col].set_title(f"{name} — waveform")
    im = axes[1, col].imshow(feat, origin="lower", aspect="auto", cmap="magma")
    axes[1, col].set_title(f"{name} — log-mel {feat.shape}")
plt.tight_layout(); plt.show()

## 3. Features

Each clip is resampled to 16 kHz mono, fixed to 4 s (loop-pad / centre-crop),
converted to a **128-bin log-mel spectrogram** (`n_fft=1024`, `hop=256`),
then **instance-normalised** (per-clip mean/var). Shape: **(128, 251)**.
**SpecAugment** (frequency/time masking) is applied during training only.

In [ ]:
from src.features import expected_num_frames
print("Feature shape (n_mels, n_frames):",
      (cfg.features.n_mels, expected_num_frames(cfg.audio, cfg.features)))

## 4. Model

`SpecNetCNN` — a compact VGG-style 2-D CNN: four `Conv-BN-ReLU ×2 → MaxPool →
Dropout` blocks (`32→64→128→128`), a global `AdaptiveAvgPool2d`, then a small
MLP head producing two logits. Global pooling makes it length-agnostic.

In [ ]:
from src.model import build_model, predict_proba

model = build_model(cfg.model).to(device)
print(f"Parameters: {count_parameters(model):,}")

# Sanity check: forward a random batch -> (B, 2) logits
dummy = torch.randn(2, 1, cfg.features.n_mels,
                    expected_num_frames(cfg.audio, cfg.features)).to(device)
with torch.no_grad():
    print("Logits shape:", model(dummy).shape)
    print("Probs shape :", predict_proba(model, dummy).shape)

## 5. Train (on diverse data)

The model ranks deepfakes well (low EER) but is **under-confident on unseen test
deepfakes**, which caps test *accuracy*. The fix is more varied training data, so
this cell trains on **multiple Fake-or-Real variants combined** (`for-norm` +
`for-rerec`), while keeping validation and test on `for-norm` (the benchmark).

Waveform augmentation stays on. More data converges fast, so **6 epochs is
plenty**. The best checkpoint (lowest validation EER) is saved to
`models/best_model.pt`.

> Combined training is ~2× the data, so epochs take longer — keep the epoch
> count low and watch the **test** result in section 6, not the validation
> number.

In [ ]:
from src.train import train
from src.dataset import scan_dataset, class_distribution

# ── Build a DIVERSE training set from multiple Fake-or-Real variants ──────────
# The FoR dataset ships 4 versions (for-norm / for-2sec / for-original /
# for-rerec). Training on more than one exposes the model to varied recording
# conditions, so it scores *unseen* deepfakes more confidently — the fix for the
# low test accuracy. We keep validation + test on for-norm (our benchmark split).
def find_all_splits(bases, max_depth=6):
    """Return {variant_name: {training:..., validation:..., testing:...}}."""
    from collections import deque
    out = {}
    dq = deque((b, 0) for b in bases if b and os.path.isdir(b))
    seen = set()
    while dq:
        d, depth = dq.popleft()
        if d in seen or depth > max_depth:
            continue
        seen.add(d)
        names = {}
        try:
            for e in os.scandir(d):
                if e.is_dir():
                    names[e.name.lower()] = e.path
        except OSError:
            continue
        if "training" in names and _is_split_dir(names["training"]):
            out[os.path.basename(d)] = names
            continue                      # this is a split root; don't go deeper
        for nm, path in names.items():
            if nm not in _CLASS_DIRS:
                dq.append((path, depth + 1))
    return out

variants = find_all_splits(["/kaggle/input"])
print("FoR variants found:", list(variants.keys()))

# Train on for-norm + for-rerec (re-recorded = channel-diverse, closest to the
# hard test conditions). Falls back to whatever variants exist.
TRAIN_ON = [v for v in variants if any(k in v.lower() for k in ("norm", "rerec"))]
TRAIN_ON = TRAIN_ON or list(variants)

train_samples = []
for v in TRAIN_ON:
    s = scan_dataset(variants[v]["training"])
    train_samples += s
    print(f"  + {v}/training: {len(s)}")
print("Combined train total:", len(train_samples),
      dict(class_distribution(train_samples)))

# Validation + test stay on for-norm (the benchmark).
norm = next((v for v in variants if "norm" in v.lower()), TRAIN_ON[0])
val_samples = scan_dataset(variants[norm]["validation"]) if variants[norm].get("validation") else None
cfg.paths.test_dir = variants[norm].get("testing") or variants[norm].get("test")
print(f"Validation/test from: {norm}  | test_dir = {cfg.paths.test_dir}")

# ── Train ────────────────────────────────────────────────────────────────────
cfg.train.num_workers = 4      # speed (overrides config if it wasn't re-uploaded)
cfg.train.epochs = 6           # more data converges fast; 6 is plenty

checkpoint = train(cfg, train_samples=train_samples, val_samples=val_samples)
print("Best epoch:", checkpoint["best_epoch"],
      "| val EER: %.2f%%" % (checkpoint["val_eer"] * 100),
      "| val acc: %.3f" % checkpoint["val_accuracy"])

In [ ]:
# Learning curves saved during training:
from IPython.display import Image, display
import os
p = os.path.join(cfg.paths.figures_dir, "training_history.png")
if os.path.exists(p):
    display(Image(filename=p))

## 6. Evaluate (the metric that matters)

Runs the trained model over the **test set** — speakers/conditions it never saw —
and computes the full metric suite (accuracy, EER, F1, per-class accuracy,
confusion matrix), saves figures, and writes `reports/performance_report.md` with
an explicit PASS/FAIL against the verification thresholds.

**This — not the 99 %+ validation number — is the score to report and trust.**

In [ ]:
from src.evaluate import evaluate

report = evaluate(cfg, cfg.paths.model_path)
print("\nPasses verification:", report.passes_verification)

In [ ]:
# Show the saved figures
from IPython.display import Image, display
import os
for fname in ["confusion_matrix.png", "roc_curve.png"]:
    fp = os.path.join(cfg.paths.figures_dir, fname)
    if os.path.exists(fp):
        display(Image(filename=fp))

## 7. Inference demo — test new audio

Load the saved checkpoint through the shared inference API and classify a clip.
This is the same code path the CLI (`python -m src.predict`) and the Streamlit
app use.

In [ ]:
from src.predict import load_bundle, predict_file

bundle = load_bundle(cfg.paths.model_path)

# Point this at any audio file you want to test:
example = first_of(1) or first_of(0)   # reuse a dataset clip for the demo
result = predict_file(bundle, example)
print(result)
print(f"\n{os.path.basename(example)} -> {result['label_name']} "
      f"(confidence {result['confidence']*100:.1f}%)")

## 8. Reproduce / next steps

- **Full training run:** set `cfg.train.epochs` to 30–50 and re-run section 5,
  or from a shell: `python -m src.train`.
- **Evaluate:** `python -m src.evaluate` → updates `reports/performance_report.md`.
- **Test a file:** `python -m src.predict path/to/clip.wav`.
- **Web app:** `streamlit run app/streamlit_app.py`.
- **Improve robustness:** add ASVspoof 2019 spoof data, tune SpecAugment, or
  increase model width/epochs. Model selection is by **validation EER**, the
  metric the brief is strictest on (≤ 12 %).

## 9. Download everything (model + reports + figures)

Bundles the trained checkpoint, reports, and figures into a single zip in
`/kaggle/working` so you can grab it all in one click from the right-hand
**Output** panel.

In [ ]:
import os
import shutil

OUT = "/kaggle/working"
zip_base = os.path.join(OUT, "deepfake_outputs")

# Stage the files to bundle into a temp folder, then zip that folder.
stage = os.path.join(OUT, "_bundle")
if os.path.isdir(stage):
    shutil.rmtree(stage)
os.makedirs(stage, exist_ok=True)

for sub in ("models", "reports"):
    src = os.path.join(OUT, sub)
    if os.path.isdir(src):
        shutil.copytree(src, os.path.join(stage, sub))

zip_path = shutil.make_archive(zip_base, "zip", stage)
shutil.rmtree(stage)

size_kb = os.path.getsize(zip_path) // 1024
print("Wrote:", zip_path)
print("Size :", size_kb, "KB")
print("Grab it from the right-hand Output panel.")